# Multimodal baseline (late fusion) — Subtask A1

Encode the meme text with a BERT-family model and the image with a ViT-family model, concatenate the two pooled embeddings, and train a small MLP head on top. Jupyter version of [`baselines/train_multimodal.py`](../train_multimodal.py).

Runs comfortably on a single mid-range GPU.

In [ ]:
import sys, os, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoImageProcessor, AutoModel, get_linear_schedule_with_warmup

TASK1 = Path.cwd().resolve()
while TASK1.name != 'task1' and TASK1.parent != TASK1:
    TASK1 = TASK1.parent
sys.path.insert(0, str(TASK1 / 'baselines'))

from io_utils import read_jsonl, write_subtask_a1_tsv
from labels import get_task

DATA = TASK1 / 'data'
SUBTASK = 'a1'
TEXT_MODEL = 'aubmindlab/bert-base-arabertv02'
IMG_MODEL = 'google/vit-base-patch16-224'
MAX_LEN = 128
EPOCHS = 3
BATCH = 16
LR_ENCODERS = 2e-5
LR_HEAD = 1e-4
SEED = 42
RUN_ID = 'arabert_vit_late_fusion'
OUT = TASK1 / 'predictions' / 'mm_a1.tsv'

for fn in (random.seed, np.random.seed, torch.manual_seed):
    fn(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Load splits

In [ ]:
spec = get_task(SUBTASK)
train_records = read_jsonl(DATA / 'splits' / 'train.jsonl')
dev_records = read_jsonl(DATA / 'splits' / 'dev.jsonl')
target_records = read_jsonl(DATA / 'splits' / 'dev_test.jsonl')
print(f'train={len(train_records)}  dev={len(dev_records)}  target={len(target_records)}')

## 2. Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
processor = AutoImageProcessor.from_pretrained(IMG_MODEL)

class MMDataset(Dataset):
    def __init__(self, records, spec, data_dir, tokenizer, processor, max_len):
        self.records = records; self.spec = spec; self.data_dir = data_dir
        self.tok = tokenizer; self.proc = processor; self.max_len = max_len
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        enc = self.tok(r.get('text') or '', padding='max_length', truncation=True,
                       max_length=self.max_len, return_tensors='pt')
        try:
            img = Image.open(self.data_dir / r['image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(0, 0, 0))
        px = self.proc(img, return_tensors='pt')['pixel_values'].squeeze(0)
        label = r.get('label')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'pixel_values': px,
            'labels': torch.tensor(-100 if label is None else self.spec.label2id[label], dtype=torch.long),
        }

def collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

train_loader = DataLoader(MMDataset(train_records, spec, DATA, tokenizer, processor, MAX_LEN),
                          batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
dev_loader = DataLoader(MMDataset(dev_records, spec, DATA, tokenizer, processor, MAX_LEN),
                        batch_size=2*BATCH, shuffle=False, collate_fn=collate, num_workers=2)
target_loader = DataLoader(MMDataset(target_records, spec, DATA, tokenizer, processor, MAX_LEN),
                           batch_size=2*BATCH, shuffle=False, collate_fn=collate, num_workers=2)
print('built loaders')

## 3. Model

In [ ]:
class LateFusion(nn.Module):
    def __init__(self, text_name, img_name, num_labels, hidden=512, dropout=0.1):
        super().__init__()
        self.text = AutoModel.from_pretrained(text_name)
        self.image = AutoModel.from_pretrained(img_name)
        td = self.text.config.hidden_size
        idim = self.image.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(td + idim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, num_labels),
        )
    def _t(self, ids, mask):
        o = self.text(input_ids=ids, attention_mask=mask)
        return getattr(o, 'pooler_output', None) if getattr(o, 'pooler_output', None) is not None else o.last_hidden_state[:, 0, :]
    def _i(self, px):
        o = self.image(pixel_values=px)
        return getattr(o, 'pooler_output', None) if getattr(o, 'pooler_output', None) is not None else o.last_hidden_state[:, 0, :]
    def forward(self, input_ids, attention_mask, pixel_values):
        return self.head(torch.cat([self._t(input_ids, attention_mask), self._i(pixel_values)], dim=-1))

model = LateFusion(TEXT_MODEL, IMG_MODEL, spec.num_labels).to(device)

enc_params = list(model.text.parameters()) + list(model.image.parameters())
head_params = list(model.head.parameters())
optimizer = torch.optim.AdamW([
    {'params': enc_params, 'lr': LR_ENCODERS, 'weight_decay': 0.01},
    {'params': head_params, 'lr': LR_HEAD, 'weight_decay': 0.01},
])
num_steps = max(len(train_loader) * EPOCHS, 1)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * num_steps), num_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

## 4. Train

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], pixel_values=batch['pixel_values'])
            loss = loss_fn(logits, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
            total += float(loss) * labels.size(0); n += labels.size(0)
    return total / max(n, 1)

best = float('inf'); best_state = None
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    dv = run_epoch(dev_loader, train=False)
    print(f'[epoch {ep}/{EPOCHS}] train_loss={tr:.4f}  dev_loss={dv:.4f}')
    if dv < best:
        best = dv
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)

## 5. Predict + write submission

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    preds = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop('labels', None)
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], pixel_values=batch['pixel_values'])
        preds.append(logits.cpu().numpy())
    return np.concatenate(preds, axis=0)

logits = predict(target_loader)
pred_ids = logits.argmax(axis=1)
rows = [(r['id'], spec.id2label[int(pi)]) for r, pi in zip(target_records, pred_ids)]
OUT.parent.mkdir(parents=True, exist_ok=True)
write_subtask_a1_tsv(rows, OUT, run_id=RUN_ID)
print('wrote', OUT)

In [ ]:
import subprocess
print(subprocess.check_output(
    [sys.executable, str(TASK1 / 'format_checker' / 'format_checker.py'),
     '--subtask', SUBTASK, '--predictions', str(OUT)],
    text=True,
))